# 单词续写思路
- 整个代码包括：数据解析加载、模型构建、损失函数构建、优化器构建、迭代训练、模型评估可视化、模型持久化等；
- 训练整体思路：token续写，基于前缀预测下一个token是什么;
- 数据解析加载：
  * 提取出训练的单词(不包含任何非字母的字符)；
  * 词典使用：三个特殊字符串 + 26个字母，不考虑大小写；
  * Tokenizer分词器构建：按照字母进行转换；
  * Dataset的构造：需要构造出token_ids和token_masks;
  * DataLoader的构造：需要注意批次填充；
- 模型构建：
  * 整体采用解码器结构，解码器输入和解码器输出之间存在一个错位，也就是上一个时刻的输出就是当前时刻的输入；可以考虑模型迁移以及模型参数恢复的相关逻辑代码；
  * 模型输入：
    - token_ids: \[bs,t] token id输入，bs个样本，每个样本有t个token，可能存在填充的情况；eg: ``` torch.tensor([[12,25,36,12], [78,56,4,0]]) ```
    - token_masks: \[bs,t] token id对应的mask信息，如果当前token为填充值，那么对应mask为0，否则为1; eg: ``` torch.tensor([[1,1,1,1], [1,1,1,0]]) ```
  * 模型输出：
    - pred_score: \[bs,t,vocab_size] 预测每个样本、每个token实际对应预测属于各个类别的置信度信息; eg:
      ```python
          torch.tensor([
              [
                  [0.1, 0.3, ..., 0.7],
                  [-0.2, 0.4, ....., -1.3],
                  [1.1, 0.1, ..., 0.9]
                  [-2.1, 3.1, ..., 4.9]
              ],
              [
                  [5.1, 3.3, ..., 2.7],
                  [-8.2, 5.4, ....., -2.3],
                  [9.1, 6.1, ..., 0.4]
                  [-6.1, 4.1, ..., 4.1]
              ],
          ])
      ```
- 损失函数：
  * 模型输出数据为预测每个样本的每个token属于各个类别的置信度，shape为: \[bs,t,vocab_size]
  * 实际样本类别含义为每个样本的每个token对应的下一个预测token id，shape为: \[bs,t], 类别取值范围为: \[0, vocab_size)
  * 分类任务，直接采用交叉熵损失函数: ```nn.CrossEntropyLoss```
- 优化器构建：
  * 可直接采用SGD优化器: optim.SGD(net.parameters(), lr=0.001)
  * PS: 也可以针对不同参数使用不同的优化器，以及采用动态学习率的更新方式等；
- 迭代训练：
  * 正常代码逻辑：直接迭代循环dataloader获取批次数据，执行前行获取前行结果，执行反向进行参数更新；
  * PS: 可以考虑提前停止的模型训练逻辑，当模型评估效果长时间没有变化的时候，提前停止模型的训练(停止条件可以更改)；
- 模型评估可视化：
  * 正常模型评估代码逻辑：采用分类指标即可，eg: 准确率、F1值等；
- 模型持久化：
  * 正常模型持久化代码逻辑：持久化last.pkl和best.pkl两个模型文件，主要持久化模型参数、优化器参数、epoch、评估效果等信息；
- 部署应用/单词续写应用：
  * 将给定的单词前缀转换为token ids;
  * 循环迭代的过程：每次预测下一个时刻的token id，并判断是否结束生成过程；
  * 将所有token ids转换为对应的token字符串，并拼接返回；

In [1]:
import re

import random
import copy
import os

from typing import List

from datetime import datetime

import json

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from tqdm import tqdm

import numpy as np

/usr/local/lib/python3.11/site-packages/torch/cuda/__init__.py:56: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [2]:
def load_json(json_file):
    with open(json_file, "r", encoding="utf-8") as reader:
        return json.load(reader)

def save_json(json_file, json_obj):
    with open(json_file, "w", encoding="utf-8") as writer:
        json.dump(json_obj, writer, indent=2, ensure_ascii=False)

def tensor_to_device(batch, device):
    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            v = v.to(device=device)
            batch[k] = v

@torch.no_grad()
def accuracy(score, labels, label_masks):
    pred_ids = torch.argmax(score, dim=-1) # [bs,t,c] -> [bs,t]
    is_equals = pred_ids == labels # [bs, t]
    is_equals = is_equals.to(dtype=torch.float32)
    label_masks = label_masks.to(dtype=torch.float32) # [bs,t]
    is_equals = is_equals * label_masks # 填充位置的值设置为0

    # 所有相等
    acc = torch.sum(is_equals) / torch.sum(label_masks).cpu()
    return acc

def is_english_only(s):
    """
        判断给定字符串是否只有英文字母字符 a-z A-Z
    """
    pattern = r"^[A-Za-z]+$"
    return bool(re.fullmatch(pattern, s))

## 数据解析加载：

### 提取单词数据

In [4]:
in_file = "./datas/text8"
out_file = "./datas/words.txt"


os.makedirs(os.path.dirname(out_file), exist_ok=True)
block_size = 4096

with open(in_file, "r", encoding="utf-8") as reader, open(out_file, "w", encoding="utf-8") as writer:
    all_words = set()
    remaining = "" # 上一个时刻的添加
    while True:
        block = reader.read(block_size)
        if not block:
            if remaining and is_english_only(remaining):
                all_words.add(remaining.lower())
            break
        text = remaining + block

        words = text.split(" ")
        remaining = words[-1] # 最后一个可能被拆开（每次获取block_size个字节的时候，可能存在将一个完成的单词截断成两个）
        words = words[:-1]

        for word in words:
            word = word.strip()
            if len(word) == 0:
                continue
            if is_english_only(word):
                all_words.add(word.lower())

    # 所有单词输出
    all_words = sorted(list(all_words))
    for word in tqdm (all_words):
        writer.writelines(f"{word}\n")
    print(f"总单词数目:{len(all_words)}")

100%|██████████| 253854/253854 [00:00<00:00, 790649.54it/s]

总单词数目:253854


In [7]:
random.seed(14) # 设置随机数种子

in_file = "./datas/words.txt"

is_min = True
train_file = "./datas/train_words.txt"
eval_file = "./datas/eval_words.txt"
if is_min:
    train_file = "./datas/train_words_min.txt"
    eval_file = "./datas/eval_words_min.txt"

train_cnt, eval_cnt = 0, 0
with open(in_file, "r", encoding="utf-8") as reader, \
    open(train_file, "w", encoding="utf-8") as train_writer, \
    open(eval_file, "w", encoding="utf-8") as eval_writer:
        for word in reader:
            if is_min and (random.random() < 0.5 or not word.startswith("h")):
                continue
            
            word = f"{word.strip()}\n"
            if random.random() < 0.2:
                eval_writer.writelines(word)
                eval_cnt += 1
            else:
                train_writer.writelines(word)
                train_cnt += 1

print(f"训练数据样本数目:{train_cnt}")
print(f"评估数据样本数目:{eval_cnt}")

训练数据样本数目:4233
评估数据样本数目:1087


### 词汇表构建

In [8]:
tokens = {
    '<PAD>': 0,
    '<UNK>': 1,
    '<END>': 2,
}
for token in list('abcdefghijklmnopqrstuvwxyz'):
    tokens[token] = len(tokens)

print(f"总token数目:{len(tokens)}")
save_json("./datas/tokens.json", tokens)

总token数目:29


### 构建Tokenizer分词器

In [3]:
class Tokenizer:
    def __init__(self, vocab_path:str, pad_token:str='<PAD>', unk_token:str='<UNK>', end_token:str='<END>'):
        self.vocabs = load_json(vocab_path)
        self.id2tokens = {_id:_token for _token,_id in self.vocabs.items()}

        self.pad_token = pad_token
        self.pad_token_id = self.vocabs[self.pad_token]
        
        self.unk_token = unk_token
        self.unk_token_id = self.vocabs[self.unk_token]
        
        self.end_token = end_token
        self.end_token_id = self.vocabs[self.end_token]

    def vocab_size(self):
        return len(self.vocabs)

    def tokenizers(self, text:str) -> List[str]:
        """
            针对给定的文本进行分词，并返回分词结果
        """
        if text is None:
            return []
        if len(text) == 0:
            return []
        return list(text.lower())

    def convert_tokens_to_ids(self, tokens:List[str]) -> List[int]:
        """
            针对给定的tokens列表中的token进行id转换
        """
        return [self.convert_token_to_id(token) for token in tokens]

    def convert_token_to_id(self, token:str) -> int:
        """
            将对应的token转换为id
        """
        return self.vocabs.get(token, self.unk_token_id)

    def build_inputs_with_special_tokens(self, token_ids:List[int]) -> List[int]:
        """
            添加特殊token
        """
        return token_ids + [self.end_token_id]

    def convert_ids_to_tokens(self, token_ids:List[int]) -> List[str]:
        """
            将token id列表转换为token字符串列表
        """
        return [self.covnert_id_to_token(token_id) for token_id in token_ids]

    def covnert_id_to_token(self, token_id:int) -> str:
        """
            将token id转换为token字符串
        """
        return self.id2tokens[token_id]

In [4]:
tokenizer = Tokenizer("./datas/tokens.json")
word = "hello"

tokens = tokenizer.tokenizers(word)
print(f"分词后结果:{tokens}")
token_ids = tokenizer.convert_tokens_to_ids(tokens)
print(f"转换的token id结果:{token_ids}")
token_ids = tokenizer.build_inputs_with_special_tokens(token_ids)
print(f"添加特殊token id后结果:{token_ids}")
tokens = tokenizer.convert_ids_to_tokens(token_ids)
print(f"恢复的token字符串列表:{tokens}")

分词后结果:['h', 'e', 'l', 'l', 'o']
转换的token id结果:[10, 7, 14, 14, 17]
添加特殊token id后结果:[10, 7, 14, 14, 17, 2]
恢复的token字符串列表:['h', 'e', 'l', 'l', 'o', '<END>']


### 构建Dataset

In [5]:
class WordGenerateDataset(Dataset):
    def __init__(self, train_path:str, tokenizer:Tokenizer):
        super().__init__()

        self.tokenizer = tokenizer

        records = []
        with open(train_path, "r", encoding="utf-8") as reader:
            for word in reader:
                word = word.strip() # 一行一个单词
                if len(word) > 30:
                    continue

                # 文本进行分词、token id转换、添加特殊token等处理
                tokens = tokenizer.tokenizers(word)
                token_ids = tokenizer.convert_tokens_to_ids(tokens)
                token_ids = tokenizer.build_inputs_with_special_tokens(token_ids)

                records.append({
                    'word': word,
                    'token_ids': token_ids
                })
        self.records = records


    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        return {
            'word': record['word'],
            'token_ids': torch.tensor(record['token_ids'], dtype=torch.int64), 
            'token_masks': torch.ones(len(record['token_ids']), dtype=torch.int64)
        }

In [6]:
ds = WordGenerateDataset("./datas/train_words_min.txt", tokenizer)
ds[10]

{'word': 'haamer',
 'token_ids': tensor([10,  3,  3, 15,  7, 20,  2]),
 'token_masks': tensor([1, 1, 1, 1, 1, 1, 1])}

### 构建DataLoader

In [7]:
def build_collate_fn(pad_token_id):
    def _collate_fn(_batch):
        _keys = _batch[0].keys() # 获取所有列
        _result = {}
        for _key in _keys:
            _value = [_item[_key] for _item in _batch]
            _result[_key] = _value

        # 进行数据填充
        _result['token_ids'] = pad_sequence(_result['token_ids'], batch_first=True, padding_value=pad_token_id)
        _result['token_masks'] = pad_sequence(_result['token_masks'], batch_first=True, padding_value=0)
        
        return _result

    return _collate_fn

In [8]:
def build_dataloader(ds, batch_size, shuffle=True):
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        collate_fn=build_collate_fn(ds.tokenizer.pad_token_id) #  聚合方法
    )

In [9]:
dataloader = build_dataloader(ds, 2)
for batch in dataloader:
    print(batch)
    break

{'word': ['humidification', 'hpf'], 'token_ids': tensor([[10, 23, 15, 11,  6, 11,  8, 11,  5,  3, 22, 11, 17, 16,  2],
        [10, 18,  8,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0]]), 'token_masks': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])}


## 模型构建

In [10]:
def generate_one(model, token_ids, max_token_len, eos_token_id):
    """
        生成代码逻辑，基于输入的token id预测后续token
            NOTE: 当前方法仅支持单个样本的生成, 隐含要求：批次大小为1

            :model: 模型对象
            :token_ids : 已知的token id列表，shape为: [1,t] 
            :max_token_len : 最多允许的token长度(原token+新生成token的最大长度)
            :eos_token_id: 结尾token id

            :return 包含输入token ids的整个预测token id列表
    """
    _, token_len = token_ids.shape
    while token_len < max_token_len:
        pred_score = model(token_ids, torch.ones_like(token_ids)) # 获取当前情况下的预测置信度
        pred_score = pred_score[:, -1, :] # 获取最后一个时刻对应的预测置信度 [bs,vocab_size]
        pred_id = torch.argmax(pred_score, dim=-1, keepdim=True) # 获取预测类别id [bs,1]

        if pred_id[0,0] == eos_token_id:
            break
        token_ids = torch.concat([token_ids, pred_id], dim=1) 
        token_len += 1
    return token_ids

In [11]:
class BaseDecoderNetwork(nn.Module):
    def forward(self, token_ids, token_masks=None):
        """
            前向过程，获取每个token对应的预测类别信息
            NOTE:
                bs: 批次样本数目，也就是一个批次中，样本的条数；
                t: 每个样本的token数目，一个批次中的多个样本可能存在填充情况；
                vocab_size: 词表大小，也就是所有的词汇数目；
            :token_ids: 解码器输入的原始token id列表，shape为: [bs,t]
            :token_masks: 解码器输入的token对应填充信息，实际token位置为1，填充位置为0，shape为: [bs,t]

            :return 返回每个样本、每个token属于各个类别的置信度, shape为: [bs,t,vocab_size]
        """
        raise NotImplementedError("待子类实现具体逻辑!")

    def generate_one(self, token_ids, max_token_len, eos_token_id):
        """
            生成代码逻辑，基于输入的token id预测后续token
            NOTE: 当前方法仅支持单个样本的生成
            :token_ids : 已知的token id列表，shape为: [1,t] 
            :max_token_len : 最多允许的token长度
            :eos_token_id: 结尾token id

            :return 包含输入token ids的整个预测token id列表
        """
        return generate_one(self, token_ids, max_token_len, eos_token_id)
        

In [12]:
class LstmDecoderNetwork(BaseDecoderNetwork):
    """
        基于LSTM结构的解码器结构
    """

    def __init__(self, vocab_size, hidden_size, layers=3):
        super().__init__()

        # 输入token embedding层
        self.embd_layer = nn.Embedding(num_embeddings=vocab_size, embedding_dim=hidden_size)

        # LSTM层
        self.lstm_layers = nn.ModuleList([
            nn.LSTM(input_size = hidden_size, hidden_size = hidden_size, batch_first = True) for _ in range(layers)
        ])

        # 决策层
        self.classify = nn.Linear(in_features=hidden_size, out_features=vocab_size)

    def forward(self, token_ids, token_masks=None):
        # 1. 提取静态特征向量 [bs,t] -> [bs,t,e]
        token_emd = self.embd_layer(token_ids)

        # 2. 使用LSTM+残差结构增加特征融合 [bs,t,e] + [bs,t,e] --> [bs,t,e]
        lstm_input = token_emd
        for lstm_layer in self.lstm_layers:
            lstm_output, _ = lstm_layer(lstm_input)
            lstm_output = lstm_output + lstm_input
            lstm_input = lstm_output

        # 3. 决策得到各个token属于各个类别的置信度
        token_score = self.classify(lstm_input)

        return token_score
    

In [13]:
torch.random.manual_seed(24) # 设置随机数种子

net = LstmDecoderNetwork(vocab_size=80, hidden_size=4)

token_ids = torch.tensor([
    [12,25,36,12], 
    [78,56,4,0]
])
token_masks = torch.tensor([
    [1,1,1,1],
    [1,1,1,0]
])
pred_token_scores = net(token_ids=token_ids, token_masks=token_masks)
print(pred_token_scores.shape)

gen_token_ids = net.generate_one(token_ids=torch.tensor([[12,25]]), max_token_len=20, eos_token_id=1)
print(gen_token_ids)
print(gen_token_ids.shape)

torch.Size([2, 4, 80])
tensor([[12, 25, 41, 15, 39, 12]])
torch.Size([1, 6])


## 损失函数

In [17]:
class TokenClassifyLossModule(nn.Module):
    def __init__(self):
        super().__init__()
        self.loss_fn = nn.CrossEntropyLoss(reduction='none')

    def forward(self, scores, token_ids, token_masks):
        """
            基于输入的token id和mask信息计算损失，进行一个错位即可
            scores: 预测置信度 [bs,t,vocab_size]
            token_ids: 实际输入模型的token id列表 [bs,t]
            masks: token对应填充信息 1表示实际值 0表示填充值 [bs,t]
        """
       
        # 错位计算
        shift_scores = scores[..., :-1, :].contiguous() # [bs,t,c] -> [bs,t-1,c]
        shift_token_ids = token_ids[..., 1:].contiguous() # [bs,t] -> [bs,t-1]
        shift_token_masks = token_masks[..., 1:].contiguous() # [bs,t] -> [bs,t-1]
   
        # 开始计算
        shift_scores = torch.permute(shift_scores, dims=(0,2,1))
        loss = self.loss_fn(shift_scores, shift_token_ids) # [bs,t-1]
        shift_token_masks = shift_token_masks.to(dtype=loss.dtype)
        loss = loss * shift_token_masks # 填充位置的损失置为0

        # return loss
        return torch.sum(loss) / torch.sum(shift_token_masks) # 均值作为损失返回

In [18]:
loss_fn = TokenClassifyLossModule()

token_ids = torch.tensor([
    [12,25,36,12], 
    [78,56,4,0]
])
token_masks = torch.tensor([
    [1,1,1,1],
    [1,1,1,0]
])
pred_token_scores = torch.rand(2, 4, 1000)

loss = loss_fn(pred_token_scores, token_ids, token_masks)
print(loss)

tensor(7.0460)


## 优化器构建

In [19]:
def build_opt(net:nn.Module, lr:float):
    sgd = optim.SGD(net.parameters(), lr=lr)

    return sgd

In [20]:
opt_fn = build_opt(net, lr=0.1)
opt_fn

SGD (
Parameter Group 0
    dampening: 0
    differentiable: False
    foreach: None
    fused: None
    lr: 0.1
    maximize: False
    momentum: 0
    nesterov: False
    weight_decay: 0
)

## 模型效果评估
- 自动评估: 高效、低成本、低精度(低可信度)
  * BLEU(Bilingual Evaluation Understudy): 计算生成文本和真实文本之间的n-gram的重叠率；
  * ROUGE(Recall-Oriented Understudy for Gisting Evaluation): 类似BLEU，但是更加关注召回率；
  * METEOR(Metric for Evaluation of Translation with Explicit ORder): 在n-gram的基础上，考虑同义词匹配、词干匹配、语序等相关信息；
  * 基于语义的评估
  * 困惑度(Perplexity)：衡量模型对真实文本的"预测难度"，主要反映生成文本的"流畅性"；
  * 多样性(Distinct-N): 计算不重复n-gram数目和n-gram总数量的占比；
- 人工评估：低效、高成本、高精度(高可信度)
  * 流畅性：生成语法是否正确、语句是否通顺；
  * 相关性：输入和输出内容是否匹配；
  * 一致性：生成内容的各个子文本之间是否逻辑自洽；
  * 多样性：是否避免重复表达；
  * 创造性：生成内容是否新颖；

## 模型训练

In [32]:
def training():
    token_path = "./datas/tokens.json"
    jit_model_path = "./output/lstm/models/model.pt"
    #train_file = "./datas/train_words_min.txt"
    #eval_file = "./datas/eval_words_min.txt"
    train_file = "./datas/train_words.txt"
    eval_file = "./datas/eval_words.txt"
    batch_size = 16
    lstm_hidden_size = 256
    lstm_layers = 2
    total_epoch = 20
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"当前运行环境为:{device}")
    
    os.makedirs(os.path.dirname(jit_model_path), exist_ok=True)
    
    # 数据加载
    tokenizer = Tokenizer(token_path)
    train_ds = WordGenerateDataset(train_file, tokenizer)
    train_dataloader = build_dataloader(train_ds, batch_size, shuffle=True)
    eval_ds = WordGenerateDataset(eval_file, tokenizer)
    eval_dataloader = build_dataloader(eval_ds, batch_size * 2, shuffle=False)
    
    # 模型加载
    net = LstmDecoderNetwork(vocab_size=tokenizer.vocab_size(), hidden_size=lstm_hidden_size, layers=lstm_layers)
    net.to(device=device)
    loss_fn = TokenClassifyLossModule()
    opt_fn = build_opt(net, lr=0.01)
    
    # 迭代训练
    for epoch in range(total_epoch):
        # 训练
        net.train()
        train_bar = tqdm(enumerate(train_dataloader), total=len(train_dataloader))
        for batch_idx, batch in train_bar:
            tensor_to_device(batch, device=device)
    
            # 前向过程
            token_ids = batch['token_ids']
            token_masks=batch['token_masks']
            pred_token_scores = net(token_ids=token_ids, token_masks=token_masks)
            loss = loss_fn(pred_token_scores, token_ids, token_masks)
    
            # 反向过程
            loss.backward()
            opt_fn.step() # 参数更新
            opt_fn.zero_grad() # 梯度重置为0
    
            # 输出评估指标
            _msg = f"Train Epoch {epoch} Batch {batch_idx} Loss:{loss.item():.3f} "
            train_bar.set_description(_msg)
    
        # 模型评估
        with torch.no_grad():
            net.eval()
            eval_bar = tqdm(enumerate(eval_dataloader), total=len(eval_dataloader))
            for batch_idx, batch in eval_bar:
                tensor_to_device(batch, device=device)
        
                # 前向过程
                token_ids = batch['token_ids']
                token_masks=batch['token_masks']
                pred_token_scores = net(token_ids=token_ids, token_masks=token_masks)
                loss = loss_fn(pred_token_scores, token_ids, token_masks)
    
                # 输出评估指标
                _msg = f"Eval Epoch {epoch} Batch {batch_idx} Loss:{loss.item():.3f} "
                eval_bar.set_description(_msg)
    
    # 模型持久化
    jit_net = torch.jit.trace(
        net.eval().cpu(), 
        (torch.randint(0, 10, (10, 128)), torch.ones((10,128)))
    )
    torch.jit.save(
        jit_net,
        jit_model_path,
    )
    print("训练完成")

In [ ]:
training()

当前运行环境为:cuda


Eval Epoch 17 Batch 491 Loss:2.652 :  30%|██▉       | 469/1587 [00:01<00:03, 321.70it/s]

## 模型推理应用

In [21]:
# 模型恢复
jit_model_path = "./output/lstm/models/model.pt"
net = torch.jit.load(jit_model_path, map_location="cpu")
print(net)

RecursiveScriptModule(
  original_name=LstmDecoderNetwork
  (embd_layer): RecursiveScriptModule(original_name=Embedding)
  (lstm_layers): RecursiveScriptModule(
    original_name=ModuleList
    (0): RecursiveScriptModule(original_name=LSTM)
    (1): RecursiveScriptModule(original_name=LSTM)
  )
  (classify): RecursiveScriptModule(original_name=Linear)
)


In [22]:
# 分词器恢复
token_path = "./datas/tokens.json"
tokenizer = Tokenizer(token_path)

In [23]:
while True:
    word = input("请输入单词前缀:")

    if word == '1':
        break

    # 分词、token id转换
    tokens = tokenizer.tokenizers(word)
    token_ids = tokenizer.convert_tokens_to_ids(tokens)
    token_ids = torch.tensor([token_ids])

    # 生成token id
    gen_token_ids = generate_one(net, token_ids=token_ids, max_token_len=20, eos_token_id=tokenizer.end_token_id)

    # token id转文本
    tokens = tokenizer.convert_ids_to_tokens(gen_token_ids[0].numpy())
    print(f"预测结果为:{''.join(tokens)}")

请输入单词前缀: h


预测结果为:hartistic


请输入单词前缀: he


预测结果为:herestic


请输入单词前缀: 1
